# Notebook 01 — What Is Machine Learning?

In this series of notebooks, we'll first build a solid understanding of machine learning fundamentals and later apply it to the task of **image segmentation**. This first notebook is more introductory and is all about understanding the ultimate goal of machine learning.

Before we point any model at an image, we need to understand the big picture: what a **model** is, and what it actually means to **train** one.

> **Note:** Unless otherwise stated, don't try to read through or understand the code. Simply run it and focus on the plots or text it produces. 

## Introduction

Almost all of machine learning is this one idea:

> We have **historical data** (say, house prices we've collected from real estate listings), and we want to predict the **future** from it (say, the price of a house we've never seen before).

To do that we build a **model**. But how do we know if a model is any good?

- We can't test a model on the future (we simply don't have it yet).
- The only evidence we have is the data we already know.
- So the most sensible bet is: *trust the model that best explains the data we already have*.

The historical data we use to build a model is called the **training data**, and the act of building a model using it is called **training**. You can think of a model as a *hypothesis*, i.e. a candidate explanation, we have for the data.

Let's make that concrete. Suppose someone hands us three candidate models for the same set of data points. Which one should we pick?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
x = np.linspace(0, 2 * np.pi, 12)
y = np.cos(x) + rng.normal(0, 0.08, size=x.shape)

xs = np.linspace(0, 2 * np.pi, 300)
candidates = [
    ("Model A:  cos(x)",           np.cos(xs)),
    ("Model B:  a straight line",  0.20 * xs - 0.6),
    ("Model C:  cos(2x)",       np.cos(2 * xs)),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, (title, curve) in zip(axes, candidates):
    ax.scatter(x, y, color="black", zorder=3, label="Data we have")
    ax.plot(xs, curve, color="tab:red", lw=2, label="Model")
    ax.set_title(title)
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

The black data points represent our training data. Look at how close each red curve comes to these points:

- **Model A** passes right through (almost) every point. It *fits (explains)* the data well.
- **Model B** is a straight line; it can't follow the up-and-down shape, so it misses most points. It can't fit the data well.
- **Model C** wiggles, but at the wrong rate, so it also misses most points. It also can't fit the data well.

Since the only information available to us is the observed data, the natural choice is to select the model that matches it most closely: **Model A**. So our core principle is: *a good model is a hypothesis that fits the training data well.* This idea is known as **empirical risk minimization (ERM)** and forms the foundation of most machine learning methods.

But what if there are 3 million models to choose from, instead of just three? To choose the best model automatically, a machine needs a number that measures how well the fit is. That number is called the **loss**.

## Loss: measuring how wrong a model is

For any data point, the model makes a **prediction**. In the above plot, each red point that make up the whole red curve is essentially a prediction for the corresponding $x$ value (the predictions usually won't be exactly right). The loss is simply a measure of how far off the predictions are from the real answers *on average*.

In the plot below, the red curve is a model whose predictions are a little off. The gray lines show the error at every point, and the blue arrow highlights one of them.

In [ ]:
rng = np.random.default_rng(1)
x = np.linspace(0, 2 * np.pi, 12)
y = np.cos(x) + rng.normal(0, 0.08, size=x.shape)

def predict(t):
    return 0.8 * np.cos(t) + 0.12

pred = predict(x)
xs = np.linspace(0, 2 * np.pi, 300)

plt.figure(figsize=(7, 4))
for xi, yi, pi in zip(x, y, pred):
    plt.plot([xi, xi], [pi, yi], color="gray", lw=1, alpha=0.7)
plt.plot(xs, predict(xs), color="tab:red", lw=2, label="Model's prediction")
plt.scatter(x, y, color="black", zorder=3, label="Actual data")

i = 3
plt.annotate("", xy=(x[i], y[i]), xytext=(x[i], pred[i]),
             arrowprops=dict(arrowstyle="<->", color="tab:purple", lw=2))
plt.text(x[i] + 0.2, (y[i] + pred[i]) / 2, "error", color="tab:purple",
         fontsize=13, fontweight="bold", va="center")

plt.legend(loc="lower left", fontsize=9)
plt.xlabel("x"); plt.ylabel("y")
plt.title("Loss ∝ the average error")
plt.show()

Notice the "proportional to" ($\propto$) symbol in the title of the plot. The loss is usually defined as some function of the errors. We'll see different **loss functions** later on. For now, we'll use the two terms interchangeably (loss $\leftrightarrow$ error).

### So what does training do?

Each point has its own loss (its own error). Add them all up and you get a *single number*, the total loss, which says how wrong the model is overall. Training is the process of automatically adjusting a model to make the total loss across many training examples as small as possible (i.e., *minimize* the total loss).

In practice, training follows a simple cycle: start with some model, measure the total loss, make small adjustments that reduce the loss, and repeat. Over time, the model *learns* to shrink the gap between its predictions and the actual outcomes. When these gaps become consistently small, we gain confidence that the model can make accurate predictions on new, unseen data.

## A model is just a function

Another useful way to think about a model is as a form of *data compression*. Imagine that the training data consists of terabytes of examples. Storing and searching through all of that data every time we need to make a prediction would be impractical. Instead, we train a model once so it captures the underlying patterns in the data. It then takes up very little space and can answer questions on its own.

Mathematically, a model is a function we'll call $f$. It takes some inputs (called **features**) and returns a prediction $y$. If we have $n$ features, we can write the model as:

$$ y = f(x_1, x_2, \ldots, x_n) $$

Say we trained a model using a dataset of 10,000 examples, and the model we trained is $f(x_1, x_2) = 3x_1 + 2x_2$. For example, asking it to predict for $x_1 = 5, x_2 = -1$ gives $f(5, -1) = 3\,(5) + 2\,(-1) = 13$.

Consider the implication of this. By storing only two numbers, which are the coefficients of $x_1$ and $x_2$ (in this case, 3 and 2), we can make predictions for any values of $x_1$ and $x_2$ (without needing to store all 10,000 examples). The model has **compressed** the entire dataset into just two numbers.

## A concrete example: house prices

Let's use a real-world-flavored example. Each house has a feature we can measure (in this example, it's the size in square feet), and a value we want to predict (in this example, it's the price). Below is a small dataset: every dot is one house, positioned by its size ($x$) and its price ($y$).

In [ ]:
rng = np.random.default_rng(1)
sqft  = np.sort(rng.integers(600, 3000, size=10))
price = 50_000 + 150 * sqft + rng.normal(0, 30_000, size=sqft.shape)

plt.figure(figsize=(6, 4))
plt.scatter(sqft, price, color="tab:blue", alpha=0.8)
plt.xlabel("size (square feet)")
plt.ylabel("price (dollars)")
plt.title("House-price dataset: each dot is one house")
plt.show()

In [ ]:
import pandas as pd
dataset = pd.DataFrame({"size (sq ft)": sqft, "price ($)": price.round().astype(int)})
dataset = dataset.sort_values("size (sq ft)").reset_index(drop=True)
dataset.index = dataset.index + 1
dataset.index.name = "house"
dataset

> **Note:** Each element in a dataset is called an **example**. In this case, each row of the table (or each dot in the plot) represents one example. 

In real problems we'd use many features simultaneously, e.g. `f(size, location, age, number_of_bedrooms, ...)`. Obviously, size alone doesn't determine price. More useful features generally means better predictions. But one feature is much easier to picture, so we'll stick with size for now.

### Predicting a house we've never seen

Now suppose someone asks: *what's the price of a 1000 sq ft house?*

As the table above shows, there is no house with exactly 1,000 square feet in our dataset, so we cannot simply look up its price. This is precisely why we train a model: we want $f$ to learn the relationship between size and price well enough to fill in such gaps. Once trained, we can simply ask the model for $f(1000)$.

Below we train the simplest possible model (a straight line) on the data, then read off its prediction at 1000 sq ft. Don't try to understand how we actually train the model yet. For now, suppose we have a magical tool that can find the best model for us.

In [ ]:
slope, intercept = np.polyfit(sqft, price, 1)

def f(x):
    return slope * x + intercept

x_query = 1000
prediction = f(x_query)
print(f"learned rule: price = {slope:.1f} * sqft + {intercept:,.0f}")
print(f"f(1000)  ->  predicted price of a 1000 sq ft house = ${prediction:,.0f}")

xs = np.linspace(500, sqft.max(), 100)
plt.figure(figsize=(6, 4))
plt.scatter(sqft, price, color="tab:blue", alpha=0.6, label="training data")
plt.plot(xs, f(xs), color="tab:red", lw=2, label="our model  f(x)")
plt.scatter([x_query], [prediction], color="black", zorder=5, s=80, label="prediction f(1000)")
plt.axvline(x_query, color="gray", ls="--", lw=1)
plt.annotate(f"f(1000) ≈ ${prediction:,.0f}", xy=(x_query, prediction),
             xytext=(x_query + 250, prediction - 70_000),
             arrowprops=dict(arrowstyle="->"))
plt.xlabel("size (square feet)"); plt.ylabel("price (dollars)")
plt.legend()
plt.title("A trained model fills the gap: predicting an unseen house")
plt.show()

Our model's prediction for the price of a 1000 sq ft house is about $207,670.

## Summary

You now have the big picture of machine learning:

- The goal is to use **training data** (**the past**) to predict new, unseen cases (**the future**).
- A **model** is a function $y = f(x_1, \dots, x_n)$: it takes **features** in and returns a **prediction**. We can also think of it as a compact summary of the data (a compressed representation).
- **Loss** measures how wrong a prediction is (the deviation from the real answer). **Training** is the process of adjusting the model to make the total loss as small as possible.
- A good model **fits** (explains) the training data well (i.e., has small total loss).